<a href="https://colab.research.google.com/github/Jkad1/btc-direction-model/blob/main/btc_direction_model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install yfinance -q
import yfinance as yf
df = yf.download("BTC-USD", start="2019-01-01", auto_adjust=True)
df.columns = df.columns.get_level_values(0)
df = df[["Close"]]
df.tail()


[*********************100%***********************]  1 of 1 completed


Price,Close
Date,
2026-09-24,84379.062500
2026-09-25,84034.921875
2026-09-26,84406.453125
2026-09-27,84458.085938
2026-09-28,83304.789062


In [5]:
df["return"] = df["Close"].pct_change()
df["ma7"] = df["Close"].rolling(7).mean() / df["Close"] - 1
df["ma30"] = df["Close"].rolling(30).mean() / df["Close"] - 1
df["vol7"] = df["return"].rolling(7).std()
df["next"] = df["return"].shift(-1)
df = df.dropna()
df["target"] = (df["next"] > 0).astype(int)


In [6]:
features = ["return", "ma7", "ma30", "vol7"]
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_train, y_train = train[features], train["target"]
X_test, y_test = test[features], test["target"]


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
models = [LogisticRegression(), RandomForestClassifier(200, max_depth=5, random_state=0)]
for m in models:
    m.fit(X_train, y_train)
    print(type(m).__name__, accuracy_score(y_test, m.predict(X_test)))
print("Always guess up:", y_test.mean())


LogisticRegression 0.5089285714285714
RandomForestClassifier 0.5035714285714286
Always guess up: 0.49464285714285716


#Predicting Bitcoin's Next-Day Direction

**Goal:** Test whether simple price-based features can predict if Bitcoin closes up or down the next day.

**Method:** Downloaded daily BTC-USD data (2019 to present) with yfinance. Built features: daily return, 7- and 30-day moving-average distance, and 7-day volatility. Trained logistic regression and a random forest, using a time-based split (oldest 80% train, newest 20% test) to avoid look-ahead bias.

**Results:** Logistic regression ~50.9%, random forest ~50.4%, versus a 49.5% "always guess up" baseline.

**Conclusion:** Both models beat the baseline by only about 1 point, so these features carry very little signal. Daily crypto moves are noisy, and this model should not be used for trading.

**Next steps:** Add trading volume, longer-term trend features, and test across different market periods.
